In [ ]:
from nba_api.stats.static import players
import pandas as pd
import re
import requests
from bs4 import BeautifulSoup

In [234]:
all_nba_players = players.get_players()
player_ids = [player['id'] for player in all_nba_players]



In [235]:
len(player_ids)

5135

In [237]:

def clean_header(h):
    match = re.search(r'\](.*?)\[/css3_tooltip\]', h)
    h = h.upper()
    return match.group(1) if match else h

def scrape_nba_page(url, year):
    r = requests.get(url)
    soup = BeautifulSoup(r.content, "html.parser")

    headers = [clean_header(th.get_text(strip=True)) for th in soup.find_all('th')]
    headers = [h for h in headers if h not in [
    "2016-2017 NBA REGULAR SEASON PLAYER STATS",
    "NBA DFS CHEAT SHEET >>",
    "2017 PLAYOFFS PLAYER STATS"
]]
    

    

    rows = []
    for tr in soup.find_all('tr'):
        cells = tr.find_all('td')
        if not cells:
            continue
        rows.append([cell.get_text(strip=True) for cell in cells])

    df = pd.DataFrame(rows, columns=headers[:len(rows[0])])
    df['Year'] = year
    return df


In [238]:
urls = {
    "2016": "https://www.nbastuffer.com/2016-2017-nba-regular-season-player-stats/",
    "2016-p" : "https://www.nbastuffer.com/2017-nba-playoffs-player-stats/",
}


url_formats =  {x : f"https://www.nbastuffer.com/{x}-{x+1}-nba-player-stats/" for x in range(2017, 2026)}
urls.update(url_formats)


In [239]:
data = [scrape_nba_page(url, year) for year,url in urls.items()]
len(data)

11

In [ ]:
rename_map = {
    'PLAYER': 'NAME',
    'FULL NAME': 'NAME',
    'TOR': 'TO%',
    'Tor%': 'TO%',
    'TOr': 'TO%',
    'TOPG': 'TOpG',
    'TPG': 'TOpG',
    'EFG%': 'eFG%',
    'MIN%': 'MP%',
}


for df in data:
    df.rename(columns=rename_map, inplace=True)

    

Index(['RANK', 'NAME', 'TEAM', 'POS', 'AGE', 'GP', 'MPG', 'MP%', 'USG%', 'TO%',
       'FTA', 'FT%', '2PA', '2P%', '3PA', '3P%', 'TS%', 'PPG', 'RPG', 'TRB%',
       'APG', 'AST%', 'SPG', 'BPG', 'VI', 'Year'],
      dtype='object')
Index(['RANK', 'NAME', 'TEAM', 'POS', 'AGE', 'GP', 'MPG', 'MP%', 'USG%', 'TO%',
       'FTA', 'FT%', '2PA', '2P%', '3PA', '3P%', 'TS%', 'PPG', 'RPG', 'TRB%',
       'APG', 'AST%', 'SPG', 'BPG', 'VI', 'Year'],
      dtype='object')
Index(['RANK', 'NAME', 'TEAM', 'POS', 'AGE', 'GP', 'MPG', 'MP%', 'USG%', 'TO%',
       'FTA', 'FT%', '2PA', '2P%', '3PA', '3P%', 'eFG%', 'TS%', 'PPG', 'RPG',
       'TRB%', 'APG', 'AST%', 'SPG', 'BPG', 'TOpG', 'VI', 'ORTG', 'DRTG',
       'Year'],
      dtype='object')
Index(['RANK', 'NAME', 'TEAM', 'POS', 'AGE', 'GP', 'MPG', 'MP%', 'USG%', 'TO%',
       'FTA', 'FT%', '2PA', '2P%', '3PA', '3P%', 'eFG%', 'TS%', 'PPG', 'RPG',
       'TRB%', 'APG', 'AST%', 'SPG', 'BPG', 'TOpG', 'VI', 'ORTG', 'DRTG',
       'Year'],
      dtype='object'

In [241]:
data[5].columns

Index(['RANK', 'NAME', 'TEAM', 'POS', 'AGE', 'GP', 'MPG', 'MP%', 'USG%', 'TO%',
       'FTA', 'FT%', '2PA', '2P%', '3PA', '3P%', 'eFG%', 'TS%', 'PPG', 'RPG',
       'TRB%', 'APG', 'AST%', 'SPG', 'BPG', 'TOpG', 'VI', 'ORTG', 'DRTG',
       'Year'],
      dtype='object')

In [263]:
df = pd.concat(data)
df = df.reset_index().drop(columns=['index', 'RANK', 'CUR', 'TOpG',	'ORTG',	'DRTG',	'P+R',	'P+A',	'P+R+A', 'eFG%'])
df.to_csv("data/player_by_year_stats.csv")